<a href="https://colab.research.google.com/github/elvisshema19-sudo/library-lines/blob/main/Library_lines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
class Book:

    def __init__(self, isbn: str, title: str, author: str, genre: str, copies: int = 1):
        self.isbn = isbn
        self.title = title
        self.author = author
        self.genre = genre
        self.total_copies = copies
        self.available_copies = copies

    def borrow_copy(self) -> bool:
        if self.available_copies > 0:
            self.available_copies -= 1
            return True
        return False

    def return_copy(self) -> None:
        if self.available_copies < self.total_copies:
            self.available_copies += 1

    def is_available(self) -> bool:
        return self.available_copies > 0

    def to_csv_row(self) -> str:
        return f"{self.isbn},{self.title},{self.author},{self.genre},{self.total_copies},{self.available_copies}"

    def __str__(self) -> str:
        status = "Available" if self.is_available() else "Unavailable"
        return (
            f"[{self.isbn}] '{self.title}' by {self.author} "
            f"({self.genre}) — {self.available_copies}/{self.total_copies} copies — {status}"
        )

In [48]:
from datetime import date


class Member:

    MAX_BORROW_LIMIT = 5

    def __init__(self, member_id: str, name: str, email: str):
        self.member_id = member_id
        self.name = name
        self.email = email
        self.join_date: str = str(date.today())
        self.borrowed_isbns: list = []

    def can_borrow(self) -> bool:
        return len(self.borrowed_isbns) < self.MAX_BORROW_LIMIT

    def add_borrowed(self, isbn: str) -> None:
        self.borrowed_isbns.append(isbn)

    def remove_borrowed(self, isbn: str) -> bool:
        if isbn in self.borrowed_isbns:
            self.borrowed_isbns.remove(isbn)
            return True
        return False

    def borrow_count(self) -> int:
        return len(self.borrowed_isbns)

    def to_csv_row(self) -> str:
        borrowed = "|".join(self.borrowed_isbns)
        return f"{self.member_id},{self.name},{self.email},{self.join_date},{borrowed}"

    def __str__(self) -> str:
        return (
            f"[{self.member_id}] {self.name} <{self.email}> "
            f"— Borrowed: {self.borrow_count()}/{self.MAX_BORROW_LIMIT}"
        )

In [49]:
from datetime import date, timedelta


class Transaction:

    LOAN_PERIOD_DAYS = 14

    def __init__(self, transaction_id: str, member_id: str, isbn: str, action: str):
        self.transaction_id = transaction_id
        self.member_id = member_id
        self.isbn = isbn
        self.action = action
        self.transaction_date: str = str(date.today())
        self.due_date: str = (
            str(date.today() + timedelta(days=self.LOAN_PERIOD_DAYS))
            if action == "borrow"
            else "N/A"
        )

    def is_overdue(self) -> bool:
        if self.action != "borrow" or self.due_date == "N/A":
            return False
        return date.today() > date.fromisoformat(self.due_date)

    def days_until_due(self) -> int:
        if self.due_date == "N/A":
            return 0
        delta = date.fromisoformat(self.due_date) - date.today()
        return delta.days

    def to_csv_row(self) -> str:
        return f"{self.transaction_id},{self.member_id},{self.isbn},{self.action},{self.transaction_date},{self.due_date}"

    def summary(self) -> str:
        overdue_note = " [OVERDUE]" if self.is_overdue() else ""
        return (
            f"TX-{self.transaction_id}: {self.action.upper()} | "
            f"Member: {self.member_id} | Book: {self.isbn} | "
            f"Date: {self.transaction_date} | Due: {self.due_date}{overdue_note}"
        )

    def __str__(self) -> str:
        return self.summary()

In [50]:
import csv
import os


class Library:

    BOOKS_FILE = "data/books.csv"
    MEMBERS_FILE = "data/members.csv"
    TRANSACTIONS_FILE = "data/transactions.csv"

    def __init__(self, name: str):
        self.name = name
        self.books: dict = {}
        self.members: dict = {}
        self.transactions: list = []
        self._tx_counter: int = 1
        self._load_all()


    def add_book(self, book: Book) -> None:
        if book.isbn in self.books:
            raise ValueError(f"Book with ISBN '{book.isbn}' already exists.")
        self.books[book.isbn] = book
        self._save_books()

    def remove_book(self, isbn: str) -> None:
        if isbn not in self.books:
            raise KeyError(f"No book found with ISBN '{isbn}'.")
        del self.books[isbn]
        self._save_books()

    def search_books(self, keyword: str) -> list:
        keyword = keyword.lower()
        return [
            b for b in self.books.values()
            if keyword in b.title.lower()
            or keyword in b.author.lower()
            or keyword in b.genre.lower()
        ]

    def list_available_books(self) -> list:
        return [b for b in self.books.values() if b.is_available()]


    def register_member(self, member: Member) -> None:
        if member.member_id in self.members:
            raise ValueError(f"Member ID '{member.member_id}' is already registered.")
        self.members[member.member_id] = member
        self._save_members()

    def remove_member(self, member_id: str) -> None:
        if member_id not in self.members:
            raise KeyError(f"No member found with ID '{member_id}'.")
        del self.members[member_id]
        self._save_members()

    def find_member(self, member_id: str) -> Member:
        if member_id not in self.members:
            raise KeyError(f"Member '{member_id}' not found.")
        return self.members[member_id]


    def borrow_book(self, member_id: str, isbn: str) -> Transaction:
        member = self.find_member(member_id)
        if isbn not in self.books:
            raise KeyError(f"Book '{isbn}' not found in catalogue.")

        book = self.books[isbn]

        if not member.can_borrow():
            raise PermissionError(
                f"Member '{member.name}' has reached the borrow limit "
                f"({Member.MAX_BORROW_LIMIT} books)."
            )
        if not book.borrow_copy():
            raise RuntimeError(
                f"No available copies of '{book.title}' at this time."
            )

        member.add_borrowed(isbn)
        tx = Transaction(str(self._tx_counter), member_id, isbn, "borrow")
        self._tx_counter += 1
        self.transactions.append(tx)

        self._save_books()
        self._save_members()
        self._save_transactions()
        return tx

    def return_book(self, member_id: str, isbn: str) -> Transaction:
        member = self.find_member(member_id)
        if isbn not in self.books:
            raise KeyError(f"Book '{isbn}' not found in catalogue.")

        book = self.books[isbn]

        if not member.remove_borrowed(isbn):
            raise ValueError(
                f"Member '{member.name}' does not have a record of borrowing '{isbn}'."
            )

        book.return_copy()
        tx = Transaction(str(self._tx_counter), member_id, isbn, "return")
        self._tx_counter += 1
        self.transactions.append(tx)

        self._save_books()
        self._save_members()
        self._save_transactions()
        return tx


    def overdue_report(self) -> list:
        return [tx for tx in self.transactions if tx.is_overdue()]

    def member_history(self, member_id: str) -> list:
        return [tx for tx in self.transactions if tx.member_id == member_id]


    def _save_books(self) -> None:
        os.makedirs("data", exist_ok=True)
        with open(self.BOOKS_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["isbn", "title", "author", "genre", "total_copies", "available_copies"])
            for book in self.books.values():
                writer.writerow([
                    book.isbn, book.title, book.author,
                    book.genre, book.total_copies, book.available_copies
                ])

    def _save_members(self) -> None:
        os.makedirs("data", exist_ok=True)
        with open(self.MEMBERS_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["member_id", "name", "email", "join_date", "borrowed_isbns"])
            for m in self.members.values():
                writer.writerow([
                    m.member_id, m.name, m.email,
                    m.join_date, "|".join(m.borrowed_isbns)
                ])

    def _save_transactions(self) -> None:
        os.makedirs("data", exist_ok=True)
        with open(self.TRANSACTIONS_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                "transaction_id", "member_id", "isbn",
                "action", "transaction_date", "due_date"
            ])
            for tx in self.transactions:
                writer.writerow([
                    tx.transaction_id, tx.member_id, tx.isbn,
                    tx.action, tx.transaction_date, tx.due_date
                ])

    def _load_all(self) -> None:
        os.makedirs("data", exist_ok=True)
        self._load_books()
        self._load_members()
        self._load_transactions()

    def _load_books(self) -> None:
        if not os.path.exists(self.BOOKS_FILE):
            return
        with open(self.BOOKS_FILE, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                book = Book(
                    row["isbn"], row["title"], row["author"],
                    row["genre"], int(row["total_copies"])
                )
                book.available_copies = int(row["available_copies"])
                self.books[book.isbn] = book

    def _load_members(self) -> None:
        if not os.path.exists(self.MEMBERS_FILE):
            return
        with open(self.MEMBERS_FILE, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                m = Member(row["member_id"], row["name"], row["email"])
                m.join_date = row["join_date"]
                m.borrowed_isbns = row["borrowed_isbns"].split("|") if row["borrowed_isbns"] else []
                self.members[m.member_id] = m

    def _load_transactions(self) -> None:
        if not os.path.exists(self.TRANSACTIONS_FILE):
            return
        with open(self.TRANSACTIONS_FILE, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                tx = Transaction(
                    row["transaction_id"], row["member_id"],
                    row["isbn"], row["action"]
                )
                tx.transaction_date = row["transaction_date"]
                tx.due_date = row["due_date"]
                self.transactions.append(tx)
        if self.transactions:
            self._tx_counter = int(self.transactions[-1].transaction_id) + 1

In [51]:
__all__ = ["Book", "Member", "Transaction", "Library"]

In [52]:
import csv
import os


class Library:

    BOOKS_FILE = "data/books.csv"
    MEMBERS_FILE = "data/members.csv"
    TRANSACTIONS_FILE = "data/transactions.csv"

    def __init__(self, name: str):
        self.name = name
        self.books: dict = {}
        self.members: dict = {}
        self.transactions: list = []
        self._tx_counter: int = 1
        self._load_all()


    def add_book(self, book: Book) -> None:
        if book.isbn in self.books:
            raise ValueError(f"Book with ISBN '{book.isbn}' already exists.")
        self.books[book.isbn] = book
        self._save_books()

    def remove_book(self, isbn: str) -> None:
        if isbn not in self.books:
            raise KeyError(f"No book found with ISBN '{isbn}'.")
        del self.books[isbn]
        self._save_books()

    def search_books(self, keyword: str) -> list:
        keyword = keyword.lower()
        return [
            b for b in self.books.values()
            if keyword in b.title.lower()
            or keyword in b.author.lower()
            or keyword in b.genre.lower()
        ]

    def list_available_books(self) -> list:
        return [b for b in self.books.values() if b.is_available()]


    def register_member(self, member: Member) -> None:
        if member.member_id in self.members:
            raise ValueError(f"Member ID '{member.member_id}' is already registered.")
        self.members[member.member_id] = member
        self._save_members()

    def remove_member(self, member_id: str) -> None:
        if member_id not in self.members:
            raise KeyError(f"No member found with ID '{member_id}'.")
        del self.members[member_id]
        self._save_members()

    def find_member(self, member_id: str) -> Member:
        if member_id not in self.members:
            raise KeyError(f"Member '{member_id}' not found.")
        return self.members[member_id]


    def borrow_book(self, member_id: str, isbn: str) -> Transaction:
        member = self.find_member(member_id)
        if isbn not in self.books:
            raise KeyError(f"Book '{isbn}' not found in catalogue.")

        book = self.books[isbn]

        if not member.can_borrow():
            raise PermissionError(
                f"Member '{member.name}' has reached the borrow limit "
                f"({Member.MAX_BORROW_LIMIT} books)."
            )
        if not book.borrow_copy():
            raise RuntimeError(
                f"No available copies of '{book.title}' at this time."
            )

        member.add_borrowed(isbn)
        tx = Transaction(str(self._tx_counter), member_id, isbn, "borrow")
        self._tx_counter += 1
        self.transactions.append(tx)

        self._save_books()
        self._save_members()
        self._save_transactions()
        return tx

    def return_book(self, member_id: str, isbn: str) -> Transaction:
        member = self.find_member(member_id)
        if isbn not in self.books:
            raise KeyError(f"Book '{isbn}' not in catalogue.")

        book = self.books[isbn]

        if not member.remove_borrowed(isbn):
            raise ValueError(
                f"Member '{member.name}' does not have a record of borrowing '{isbn}'."
            )

        book.return_copy()
        tx = Transaction(str(self._tx_counter), member_id, isbn, "return")
        self._tx_counter += 1
        self.transactions.append(tx)

        self._save_books()
        self._save_members()
        self._save_transactions()
        return tx


    def overdue_report(self) -> list:
        return [tx for tx in self.transactions if tx.is_overdue()]

    def member_history(self, member_id: str) -> list:
        return [tx for tx in self.transactions if tx.member_id == member_id]


    def _save_books(self) -> None:
        os.makedirs("data", exist_ok=True)
        with open(self.BOOKS_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["isbn", "title", "author", "genre", "total_copies", "available_copies"])
            for book in self.books.values():
                writer.writerow([
                    book.isbn, book.title, book.author,
                    book.genre, book.total_copies, book.available_copies
                ])

    def _save_members(self) -> None:
        os.makedirs("data", exist_ok=True)
        with open(self.MEMBERS_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["member_id", "name", "email", "join_date", "borrowed_isbns"])
            for m in self.members.values():
                writer.writerow([
                    m.member_id, m.name, m.email,
                    m.join_date, "|".join(m.borrowed_isbns)
                ])

    def _save_transactions(self) -> None:
        os.makedirs("data", exist_ok=True)
        with open(self.TRANSACTIONS_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                "transaction_id", "member_id", "isbn",
                "action", "transaction_date", "due_date"
            ])
            for tx in self.transactions:
                writer.writerow([
                    tx.transaction_id, tx.member_id, tx.isbn,
                    tx.action, tx.transaction_date, tx.due_date
                ])

    def _load_all(self) -> None:
        os.makedirs("data", exist_ok=True)
        self._load_books()
        self._load_members()
        self._load_transactions()

    def _load_books(self) -> None:
        if not os.path.exists(self.BOOKS_FILE):
            return
        with open(self.BOOKS_FILE, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                book = Book(
                    row["isbn"], row["title"], row["author"],
                    row["genre"], int(row["total_copies"])
                )
                book.available_copies = int(row["available_copies"])
                self.books[book.isbn] = book

    def _load_members(self) -> None:
        if not os.path.exists(self.MEMBERS_FILE):
            return
        with open(self.MEMBERS_FILE, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                m = Member(row["member_id"], row["name"], row["email"])
                m.join_date = row["join_date"]
                m.borrowed_isbns = row["borrowed_isbns"].split("|") if row["borrowed_isbns"] else []
                self.members[m.member_id] = m

    def _load_transactions(self) -> None:
        if not os.path.exists(self.TRANSACTIONS_FILE):
            return
        with open(self.TRANSACTIONS_FILE, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                tx = Transaction(
                    row["transaction_id"], row["member_id"],
                    row["isbn"], row["action"]
                )
                tx.transaction_date = row["transaction_date"]
                tx.due_date = row["due_date"]
                self.transactions.append(tx)
        if self.transactions:
            self._tx_counter = int(self.transactions[-1].transaction_id) + 1

In [53]:

def seed(library: Library) -> None:

    sample_books = [
        Book("978-0-06-112008-4", "To Kill a Mockingbird", "Harper Lee", "Fiction", 3),
        Book("978-0-7432-7356-5", "1984", "George Orwell", "Dystopian", 2),
        Book("978-0-14-028329-7", "The Great Gatsby", "F. Scott Fitzgerald", "Fiction", 2),
        Book("978-0-316-76948-0", "The Catcher in the Rye", "J.D. Salinger", "Fiction", 1),
        Book("978-0-452-28423-4", "Brave New World", "Aldous Huxley", "Dystopian", 3),
        Book("978-0-14-303943-3", "Crime and Punishment", "Fyodor Dostoevsky", "Classic", 2),
        Book("978-0-06-093546-9", "To the Lighthouse", "Virginia Woolf", "Modernist", 1),
        Book("978-0-7432-7357-2", "Animal Farm", "George Orwell", "Satire", 4),
    ]

    sample_members = [
        Member("M001", "Alice Müller", "alice@example.com"),
        Member("M002", "Bob Schmidt", "bob@example.com"),
        Member("M003", "Clara Weber", "clara@example.com"),
    ]

    for book in sample_books:
        try:
            library.add_book(book)
            print(f"  Added: {book.title}")
        except ValueError:
            print(f"  Skipped (already exists): {book.title}")

    for member in sample_members:
        try:
            library.register_member(member)
            print(f"  Registered: {member.name}")
        except ValueError:
            print(f"  Skipped (already exists): {member.name}")

    print("\n  Seed complete.")


lib = Library("Gisma Library")
seed(lib)


  Skipped (already exists): To Kill a Mockingbird
  Skipped (already exists): 1984
  Skipped (already exists): The Great Gatsby
  Skipped (already exists): The Catcher in the Rye
  Skipped (already exists): Brave New World
  Skipped (already exists): Crime and Punishment
  Skipped (already exists): To the Lighthouse
  Skipped (already exists): Animal Farm
  Skipped (already exists): Alice Müller
  Skipped (already exists): Bob Schmidt
  Skipped (already exists): Clara Weber

  Seed complete.
